# Property Search Pipeline — Iceberg Viewer
Run simple PySpark SQL queries against the `rental_property` Iceberg table.

## 1. Setup — SparkSession with Iceberg

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IcebergViewer")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2",
    )
    # Iceberg catalog
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type",      "hadoop")
    .config("spark.sql.catalog.local.warehouse", "data/iceberg_warehouse")
    # Iceberg SQL extensions
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory",          "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession ready:", spark.version)

## 2. Viewer — Load the Iceberg Table

In [ ]:
# Load rental_property Iceberg table into a DataFrame
df = spark.table("local.property_db.rental_property")

print("Schema:")
df.printSchema()
print(f"Total rows: {df.count()}")

## 3. Simple Queries — PySpark SQL

In [ ]:
# Register as temp view for SQL queries
df.createOrReplaceTempView("rental_property")

In [ ]:
# Q1. Preview first 10 rows
spark.sql("""
    SELECT *
    FROM rental_property
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Q2. Row count per country
spark.sql("""
    SELECT   country_code,
             COUNT(*) AS total_properties
    FROM     rental_property
    GROUP BY country_code
    ORDER BY total_properties DESC
""").show()

In [ ]:
# Q3. Filter by data_quality_flag = GOOD
spark.sql("""
    SELECT   id, property_name, country_code, usd_price, data_quality_flag
    FROM     rental_property
    WHERE    data_quality_flag = 'GOOD'
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Q4. Average star_rating and review_score per country
spark.sql("""
    SELECT   country_code,
             ROUND(AVG(star_rating),  2) AS avg_star_rating,
             ROUND(AVG(review_score), 2) AS avg_review_score
    FROM     rental_property
    GROUP BY country_code
    ORDER BY avg_review_score DESC
""").show()

In [ ]:
# Q5. Properties with usd_price defaulted to 0.0
spark.sql("""
    SELECT   id, property_name, country_code, usd_price
    FROM     rental_property
    WHERE    usd_price = 0.0
""").show(truncate=False)

In [ ]:
# Q6. Top 10 properties by review_score
spark.sql("""
    SELECT   id, property_name, country_code,
             star_rating, review_score
    FROM     rental_property
    ORDER BY review_score DESC
    LIMIT 10
""").show(truncate=False)

## 4. Iceberg Table Features

In [ ]:
# Iceberg — Table history
spark.sql("""
    SELECT *
    FROM local.property_db.rental_property.history
""").show(truncate=False)

In [ ]:
# Iceberg — Table snapshots
spark.sql("""
    SELECT *
    FROM local.property_db.rental_property.snapshots
""").show(truncate=False)

In [ ]:
# Iceberg — Table partitions
spark.sql("""
    SELECT *
    FROM local.property_db.rental_property.partitions
""").show(truncate=False)

In [ ]:
# Iceberg — Table files (data file metadata)
spark.sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM local.property_db.rental_property.files
""").show(truncate=False)

## 5. Stop SparkSession

In [ ]:
spark.stop()
print("SparkSession stopped.")